# Text classification using RNNs

## Pre-processing

In [1]:
import pandas as pd

data = pd.read_csv('../data/petitions.csv')
data.head()

,id,public_petition_text,reason_category
0,3168490,снег на дороге,Благоустройство
1,3219678,очистить кабельный киоск от рекламы,Благоустройство
2,2963920,"Просим убрать все деревья и кустарники, которы...",Благоустройство
3,3374910,Неудовлетворительное состояние парадной - надп...,Содержание МКД
4,3336285,Граффити,Благоустройство


In [2]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 59889 entries, 0 to 59888
Data columns (total 3 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id                    59889 non-null  int64
 1   public_petition_text  59889 non-null  str  
 2   reason_category       59889 non-null  str  
dtypes: int64(1), str(2)
memory usage: 1.4 MB


In [3]:
x = data["public_petition_text"]
y = data["reason_category"]

In [4]:
import re
import nltk
import pymorphy3
from razdel import tokenize
from nltk.corpus import stopwords

nltk.download('stopwords')
russian_stopwords = set(stopwords.words('russian'))

morph = pymorphy3.MorphAnalyzer()

def preprocess_text(text):

    text = text.lower()

    # Удаление URL
    text = re.sub(r'http\S+|www\.\S+', '', text)

    # Удаление email
    text = re.sub(r'\S+@\S+', '', text)

    # Замена числительных на специальный токен
    text = re.sub(r'\d+', 'NUM', text)

    # Токенизация через razdel
    tokens = [token.text for token in tokenize(text)]

    # Убирает точки, запятые, скобки, тире, кавычки и смайлики
    clean_tokens = [
        t for t in tokens 
        if re.search(r'[а-яёa-z]', t) or t == 'NUM'
    ]

    # Лемматизация
    lemmatized_words = [morph.parse(word)[0].normal_form for word in clean_tokens]

    # Удаление стоп-слов
    filtered_tokens = [word for word in lemmatized_words if word not in russian_stopwords]

    return filtered_tokens

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/irinaaristova/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


### Dataset tokenization

In [5]:
from tqdm.auto import tqdm

tqdm.pandas()

processed_tokens = x.progress_apply(preprocess_text)

valid_mask = processed_tokens.apply(len) > 0

processed_tokens = processed_tokens[valid_mask].reset_index(drop=True)

y = y[valid_mask].reset_index(drop=True)

  0%|          | 0/59889 [00:00<?, ?it/s]

### Target encoding

In [6]:
y.value_counts()

reason_category
Благоустройство                                                                     34769
Содержание МКД                                                                      14461
Нарушение правил пользования общим имуществом                                        2170
Незаконная информационная и (или) рекламная конструкция                              1831
Фасад                                                                                1483
Повреждения или неисправность элементов уличной инфраструктуры                       1164
Кровля                                                                                825
Водоснабжение                                                                         809
Состояние рекламных или информационных конструкций                                    673
Санитарное состояние                                                                  434
Центральное отопление                                                               

In [7]:
dict_y = {}
for i, val in enumerate(y.value_counts().index, start=0):
    dict_y[val] = i

dict_y

{'Благоустройство': 0,
 'Содержание МКД': 1,
 'Нарушение правил пользования общим имуществом': 2,
 'Незаконная информационная и (или) рекламная конструкция': 3,
 'Фасад': 4,
 'Повреждения или неисправность элементов уличной инфраструктуры': 5,
 'Кровля': 6,
 'Водоснабжение': 7,
 'Состояние рекламных или информационных конструкций': 8,
 'Санитарное состояние': 9,
 'Центральное отопление': 10,
 'Подвалы': 11,
 'Незаконная реализация товаров с торгового оборудования (прилавок, ящик, с земли)': 12,
 'Нарушение порядка пользования общим имуществом': 13,
 'Водоотведение': 14}

In [8]:
y_encoded = y.map(dict_y)

### Conversion to tensors

In [9]:
import torch
from navec import Navec

path = 'navec_hudlit_v1_12B_500K_300d_100q.tar'
navec = Navec.load(path)

UNK_ID = navec.vocab.unk_id
PAD_ID = navec.vocab.pad_id

max_len = processed_tokens.apply(len).max()

def tokens_to_ids(tokens):
    ids = [navec.vocab.get(word, UNK_ID) for word in tokens]
    return ids + [PAD_ID] * (max_len - len(ids))

x_indices = processed_tokens.apply(tokens_to_ids)

device = torch.device("mps")
x_tensor = torch.tensor(list(x_indices), dtype=torch.long)
y_tensor = torch.tensor(list(y_encoded), dtype=torch.long)

### Separating data

In [10]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x_tensor, y_tensor, test_size=0.15, random_state=81, shuffle=True)

### DataLoader

In [11]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(x_train, y_train)
test_dataset = TensorDataset(x_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

### Embeddings for Russian language

In [12]:
import numpy as np

vocab_size = len(navec.vocab.words)
emded_dim = 300
embedding_matrix = np.zeros((vocab_size, emded_dim))

for word in navec.vocab.words:
    idx = navec.vocab[word]
    embedding_matrix[idx] = navec[word]

## Attention layers

In [13]:
import torch.nn as nn

class AttentionLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, output, hidden):
        # Умножаем [batch, seq, hidden] на [batch, hidden, 1]
        e = torch.bmm(output, hidden.unsqueeze(2)).squeeze(2)
        alpha = torch.nn.functional.softmax(e, dim=1)

        c = torch.cat([hidden, torch.bmm(alpha.unsqueeze(1), output).squeeze(1)], dim=1)
        return c

## RNN (Recurrent Neural Networks)

In [14]:
class RNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=True
        )

        self.rnn = nn.RNN(
            input_size=300,
            hidden_size=128,
            num_layers=2,
            batch_first=True
        )

        self.attention_layer = AttentionLayer()

        self.fc = nn.Linear(256, 15)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.rnn(embedded)
        final_hidden = hidden[-1]
        c = self.attention_layer(output, final_hidden)
        logits = self.fc(c)
        
        return logits

In [15]:
import torch.optim as optim

model = RNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [16]:
from sklearn.metrics import f1_score

epochs = 5

for epoch in range(epochs):
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()

        outputs = model(batch_x)

        loss = criterion(outputs, batch_y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predicted = outputs.argmax(dim=1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(batch_y.cpu().numpy())
        
    epoch_f1 = f1_score(all_targets, all_preds, average='weighted')
    
    print(f"Эпоха {epoch+1}/{epochs} | F1: {epoch_f1:.4f}")

Эпоха 1/5 | F1: 0.4379
Эпоха 2/5 | F1: 0.4701
Эпоха 3/5 | F1: 0.4817
Эпоха 4/5 | F1: 0.4800
Эпоха 5/5 | F1: 0.4707


### LSTM (Long-Short Term Memory)

In [18]:
class LSTM(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=False
        )

        self.lstm = nn.LSTM(
            input_size=300,
            hidden_size=128,
            num_layers=2,
            batch_first=True
        )

        self.attention_layer = AttentionLayer()

        self.fc = nn.Linear(256, 15)

    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)
        final_hidden = hidden[-1]
        c = self.attention_layer(output, final_hidden)
        logits = self.fc(c)
        
        return logits

In [19]:
model_lstm = LSTM().to(device)

optimizer = optim.Adam(model_lstm.parameters(), lr=0.01)

In [20]:
epochs = 5

for epoch in range(epochs):
    model_lstm.train()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()

        outputs = model_lstm(batch_x)

        loss = criterion(outputs, batch_y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predicted = outputs.argmax(dim=1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(batch_y.cpu().numpy())
        
    epoch_f1 = f1_score(all_targets, all_preds, average='weighted')
    
    print(f"Эпоха {epoch+1}/{epochs} | F1: {epoch_f1:.4f}")

Эпоха 1/5 | F1: 0.8119
Эпоха 2/5 | F1: 0.9266
Эпоха 3/5 | F1: 0.9426
Эпоха 4/5 | F1: 0.9500
Эпоха 5/5 | F1: 0.9537


### GRU (Gated Recurrent Unit)

In [21]:
class GRU(nn.Module):
    def __init__(self):
        super().__init__()

        self.embedding = nn.Embedding.from_pretrained(
            torch.tensor(embedding_matrix, dtype=torch.float32),
            freeze=True
        )

        self.gru = nn.GRU(
            input_size=300,
            hidden_size=128,
            num_layers=2,
            batch_first=True
        )

        self.attention_layer = AttentionLayer()

        self.fc = nn.Linear(256, 15)

    def forward(self, x):
        embedded = self.embedding(x)
        output, hidden = self.gru(embedded)
        final_hidden = hidden[-1]
        c = self.attention_layer(output, final_hidden)
        logits = self.fc(c)
        
        return logits

In [22]:
model_gru = GRU().to(device)

optimizer = optim.Adam(model_gru.parameters(), lr=0.01)

In [23]:
epochs = 5

for epoch in range(epochs):
    model_gru.train()
    total_loss = 0
    all_preds = []
    all_targets = []

    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()

        outputs = model_gru(batch_x)

        loss = criterion(outputs, batch_y)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

        predicted = outputs.argmax(dim=1)
        all_preds.extend(predicted.cpu().numpy())
        all_targets.extend(batch_y.cpu().numpy())
        
    epoch_f1 = f1_score(all_targets, all_preds, average='weighted')
    
    print(f"Эпоха {epoch+1}/{epochs} | F1: {epoch_f1:.4f}")

Эпоха 1/5 | F1: 0.8653
Эпоха 2/5 | F1: 0.9227
Эпоха 3/5 | F1: 0.9237
Эпоха 4/5 | F1: 0.8552
Эпоха 5/5 | F1: 0.8795


### Сomparison of metrics

| Эпоха | RNN | LSTM | GRU |
| :--- | :--- | :--- | :--- |
| 1 | 0.4379 | 0.8119 | 0.8653 |
| 2 | 0.4701 | 0.9266 | 0.9227 |
| 3 | **0.4817** | 0.9426 | **0.9237** |
| 4 | 0.4800 | 0.9500 | 0.8552 |
| 5 | 0.4707 | **0.9537** | 0.8795 |

* **Модель LSTM** с интегрированным слоем внимания продемонстрировала самую высокую эффективность и стабильный рост, достигнув абсолютного максимума F1 в 0.9537 на пятой эпохе.

* **Модель GRU** показала уверенный старт, зафиксировав пиковое значение 0.9237 на третьей эпохе. Однако на последующих шагах наблюдается деградация метрики (до 0.8552 на четвертой эпохе), что свидетельствует о начавшемся переобучении.

* **Модель RNN** по-прежнему показывает низкие результаты, достигнув лишь 0.4817. Это подтверждает, что базовая рекуррентная архитектура не справляется с проблемой затухающего градиента на длинных текстах, и даже добавление слоя внимания не может компенсировать этот фундаментальный недостаток.